# 🔬 KLA Hackathon — NAFNet-SR Image Restoration
**AI-Based Restoration of Degraded Semiconductor Inspection Images**

### Before running:
1. Go to `Runtime → Change runtime type → T4 GPU` → Save
2. Upload `train.zip` and `test_noisyLR.zip` to a folder called **`kla_data`** in your Google Drive root
3. Then click `Runtime → Run all` — everything is pre-configured!

**Expected training time:** ~2h (T4 GPU) | ~45min (A100)

In [ ]:
# CELL 1: Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# CELL 3: Clone repo & install dependencies
import os
REPO_URL = 'https://github.com/norriy0u/kla-image-restoration.git'
REPO_DIR = '/content/kla_restoration'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}
!pip install -r requirements.txt -q
print('✓ Setup complete!')

In [ ]:
# CELL 4: Extract dataset from Google Drive & auto-detect paths
import os, glob, subprocess

DRIVE_DATA = '/content/drive/MyDrive/kla_data'
LOCAL_DATA = '/content/kla_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# ── Step 1: Extract zips if not already done ──────────────────
train_zip = os.path.join(DRIVE_DATA, 'train.zip')
test_zip  = os.path.join(DRIVE_DATA, 'test_noisyLR.zip')

if not os.path.exists(train_zip):
    raise FileNotFoundError(
        f'train.zip not found at {train_zip}\n'
        'Please upload train.zip and test_noisyLR.zip to My Drive/kla_data/'
    )

# Extract with verbose output so we can see actual folder names
print('Extracting train.zip (this may take 1-2 minutes)...')
!unzip -o {train_zip} -d {LOCAL_DATA}/ | tail -5
print('Extracting test_noisyLR.zip...')
!unzip -o {test_zip} -d {LOCAL_DATA}/ | tail -5

# ── Step 2: Auto-detect actual folder structure ────────────────
print('\n── Scanning extracted directory structure ──')
for root, dirs, files in os.walk(LOCAL_DATA):
    level = root.replace(LOCAL_DATA, '').count(os.sep)
    indent = '  ' * level
    npy_count = len([f for f in files if f.endswith('.npy')])
    print(f'{indent}{os.path.basename(root)}/ ({npy_count} .npy files)')
    if level >= 3:
        break

# ── Step 3: Find GT and NoisyLR directories automatically ─────
def find_dir_with_npy(base, min_files=100):
    """Recursively find directories containing many .npy files."""
    results = []
    for root, dirs, files in os.walk(base):
        npys = [f for f in files if f.endswith('.npy')]
        if len(npys) >= min_files:
            results.append((len(npys), root))
    return sorted(results, reverse=True)

all_npy_dirs = find_dir_with_npy(LOCAL_DATA)
print('\n── Directories with .npy files ──')
for count, path in all_npy_dirs:
    print(f'  {count:4d} files  →  {path}')

# Identify GT and LR directories by name
GT_DIR   = next((p for _, p in all_npy_dirs if 'GT' in os.path.basename(p) or 'gt' in os.path.basename(p).lower()), None)
LR_DIR   = next((p for _, p in all_npy_dirs if 'Noisy' in p or 'noisy' in p.lower()), None)
TEST_DIR = next((p for _, p in all_npy_dirs if 'test' in p.lower() and ('Noisy' in p or 'noisy' in p.lower())), None)

# Fallback: if train LR and test LR share a path, discriminate by count
if LR_DIR and TEST_DIR and LR_DIR == TEST_DIR:
    noisy_dirs = [(c,p) for c,p in all_npy_dirs if 'Noisy' in p or 'noisy' in p.lower()]
    LR_DIR   = noisy_dirs[0][1] if len(noisy_dirs) > 0 else LR_DIR
    TEST_DIR = noisy_dirs[1][1] if len(noisy_dirs) > 1 else None

print(f'\nGT_DIR   = {GT_DIR}')
print(f'LR_DIR   = {LR_DIR}')
print(f'TEST_DIR = {TEST_DIR}')

# Final counts
gt_count   = len(glob.glob(os.path.join(GT_DIR,   '*.npy'))) if GT_DIR   else 0
lr_count   = len(glob.glob(os.path.join(LR_DIR,   '*.npy'))) if LR_DIR   else 0
test_count = len(glob.glob(os.path.join(TEST_DIR, '*.npy'))) if TEST_DIR else 0
print(f'\nGT files:    {gt_count}')
print(f'LR files:    {lr_count}')
print(f'Test files:  {test_count}')

if gt_count > 0 and lr_count > 0:
    print('✓ Dataset ready!')
else:
    raise RuntimeError('Could not find dataset files — check extraction above.')

In [ ]:
# CELL 5: Quick data inspection
import numpy as np
import matplotlib.pyplot as plt
import glob, os

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))

print(f'GT files found:    {len(gt_files)}')
print(f'LR files found:    {len(lr_files)}')
assert len(gt_files) > 0, f'No .npy files in GT_DIR={GT_DIR}'
assert len(lr_files) > 0, f'No .npy files in LR_DIR={LR_DIR}'

# Sample 4 evenly spaced images
n = len(gt_files)
indices = [0, n//3, 2*n//3, n-1]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, idx in enumerate(indices):
    gt = np.load(gt_files[idx])
    lr = np.load(lr_files[idx])
    lr_vis = np.clip(lr, 0, 1)

    axes[0][col].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0][col].set_title(f'GT #{idx} ({gt.shape[0]}×{gt.shape[1]})\nrange=[{gt.min():.2f},{gt.max():.2f}]', fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].imshow(lr_vis, cmap='gray', vmin=0, vmax=1)
    axes[1][col].set_title(
        f'NoisyLR #{idx} ({lr.shape[0]}×{lr.shape[1]})\n'
        f'max={lr.max():.2f} ← speckle overflow: {lr.max()>1.0}', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Sample Training Pairs — Top: GT, Bottom: NoisyLR (degraded)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: /content/sample_pairs.png')

In [ ]:
# CELL 6: Train! (~2h on T4, ~45min on A100)
# Batch size: 16 for A100, 8 for T4
BATCH_SIZE = 8
EPOCHS = 200

!python train.py \
    --gt_dir {GT_DIR} \
    --lr_dir {LR_DIR} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers 2 \
    --val_fraction 0.1 \
    --patch_size_gt 256 \
    --model_variant base \
    --weights_dir ./weights \
    --log_dir ./logs

In [ ]:
# CELL 7: TensorBoard
%load_ext tensorboard
%tensorboard --logdir ./logs

In [ ]:
# CELL 8: Evaluate on validation set (PSNR / SSIM / LPIPS)
import os, sys, json
sys.path.insert(0, '.')
os.makedirs('/content/val_outputs', exist_ok=True)

!python evaluate.py \
    --input_dir {LR_DIR} \
    --output_dir /content/val_outputs \
    --gt_dir {GT_DIR} \
    --weights ./weights/best_model.pt \
    --batch_size 8

with open('/content/val_outputs/metrics.json') as f:
    m = json.load(f)
print('\n=== VALIDATION METRICS ===')
for k, v in m.items():
    print(f'  {k}: {v}')

In [ ]:
# CELL 9: Run inference on official test set
import os, json
os.makedirs('/content/test_outputs', exist_ok=True)

!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs \
    --weights ./weights/best_model.pt \
    --batch_size 8

with open('/content/test_outputs/metrics.json') as f:
    print('=== TEST INFERENCE STATS ===')
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# CELL 10: Visualise Before → After → GT (for PPT Slide 6)
import numpy as np
import matplotlib.pyplot as plt
import glob, os

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))

# Pick 4 samples with heaviest speckle (highest max pixel value)
sample_pool = lr_files[:min(300, len(lr_files))]
lr_maxvals  = sorted([(np.load(f).max(), f) for f in sample_pool], reverse=True)
sample_lr   = [f for _, f in lr_maxvals[:4]]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
col_titles = ['NoisyLR Input (128×128)', 'NAFNet-SR Output (256×256)', 'Ground Truth (256×256)']
col_colors = ['#e74c3c', '#2ecc71', '#3498db']

for row, lr_path in enumerate(sample_lr):
    stem    = os.path.splitext(os.path.basename(lr_path))[0]
    gt_path = os.path.join(GT_DIR, f'{stem}.npy')
    out_npy = f'/content/val_outputs/{stem}.npy'

    lr_arr  = np.load(lr_path)
    gt_arr  = np.load(gt_path) if os.path.exists(gt_path) else np.zeros((256,256))
    out_arr = np.load(out_npy)  if os.path.exists(out_npy)  else np.zeros((256,256))

    for col, (arr, title, color) in enumerate(zip([lr_arr, out_arr, gt_arr], col_titles, col_colors)):
        axes[row][col].imshow(np.clip(arr, 0, 1), cmap='gray', vmin=0, vmax=1)
        axes[row][col].set_title(
            f'{title}\n[{arr.min():.2f}, {arr.max():.2f}]',
            fontsize=10, color=color, fontweight='bold')
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'Sample {stem}', fontsize=10, rotation=90, labelpad=15)

plt.suptitle('NAFNet-SR: Degraded Input → Restored → Ground Truth', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: /content/before_after_comparison.png  ← use this in PPT Slide 6!')

In [ ]:
# CELL 11: Save weights + outputs to Google Drive
import shutil, os
DRIVE_OUT = '/content/drive/MyDrive/kla_submission'
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy('./weights/best_model.pt',              f'{DRIVE_OUT}/best_model.pt')
shutil.copytree('/content/test_outputs',            f'{DRIVE_OUT}/test_outputs',            dirs_exist_ok=True)
shutil.copy('/content/before_after_comparison.png', f'{DRIVE_OUT}/before_after_comparison.png')
shutil.copy('/content/sample_pairs.png',            f'{DRIVE_OUT}/sample_pairs.png')
print(f'✓ All saved to Drive: {DRIVE_OUT}')

In [ ]:
# CELL 12: [OPTIONAL] TTA — better quality, 4x slower
import os, json
os.makedirs('/content/test_outputs_tta', exist_ok=True)
!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs_tta \
    --weights ./weights/best_model.pt \
    --tta \
    --batch_size 1
with open('/content/test_outputs_tta/metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))